# Stage 08 — DPO reranker training

**Track A (Buse) · Stage 8 of 10**

| | |
|---|---|
| **Input** | `data/preference_pairs_B.jsonl` |
| **Output** | A tuned cross-encoder in `models/reranker_dpo_B`, plus an MLflow run |
| **Promotes to** | `src/research_assistant/reranker/train_B.py`, `scripts/train_reranker_B.py` |
| **Config** | `configs/reranker_B.yaml` → `model:`, `train:` |
| **Install** | Needs the heavy extra: `pip install -e ".[train]"` |

## Where this sits in the pipeline

The candidate set from stage 04 is ordered by fusion rank, which knows nothing about
the query beyond term overlap and embedding proximity. A cross-encoder reads the query
and the chunk *together*, so it can judge relevance rather than similarity. That is
why it wins, and why it is too slow to run over the whole corpus and only over 20
candidates.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Base model | `ms-marco-MiniLM-L-6-v2` | `bge-reranker-base`, train from scratch | Already trained for ranking, 22M parameters, fine-tunes on CPU in minutes. Starting from a ranking model means the preference data only has to shift a decision boundary, not teach the task. |
| Objective | DPO | Pairwise hinge, pointwise regression | The data is preferences, not scores, and DPO takes preferences directly. A hinge loss is the simpler classical answer and is worth a ledger row as the ablation. |
| Beta | 0.1 | 0.05, 0.5 | Controls how far the tuned model may drift from the reference. Low beta stays close and is the safe default on a few hundred pairs. |
| Epochs | 1 | 3+ | With a few hundred pairs, more epochs memorise the judge rather than learn ranking. Watch the gate, not the loss curve. |
| Success criterion | Beats the baseline on the stage 05 set | Training loss goes down | Loss going down proves the model fit the judge. Only the held-out set says whether that helped. |

**Expect this to fail the first time.** A reranker trained on a few hundred noisy
pairs frequently scores below the baseline. That is a legitimate result, it belongs in
the report, and the gate exists precisely to stop it shipping.

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json
from pathlib import Path

rcfg = load_cfg("reranker")
pairs = [json.loads(l) for l in resolve(rcfg["pairs"]["out_path"])
         .read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(pairs), "pairs")

In [ ]:
from datasets import Dataset

# DPO expects prompt / chosen / rejected. For a cross-encoder ranker the prompt is the
# query and the completions are the two candidate passages.
ds = Dataset.from_list([
    dict(prompt=p["query"], chosen=p["chosen"], rejected=p["rejected"]) for p in pairs
])
ds = ds.train_test_split(test_size=0.1, seed=rcfg["train"]["seed"])
print(ds)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments
from trl import DPOTrainer, DPOConfig

m, t = rcfg["model"], rcfg["train"]
tok = AutoTokenizer.from_pretrained(m["base"])
model = AutoModelForSequenceClassification.from_pretrained(m["base"], num_labels=m["num_labels"])
ref_model = AutoModelForSequenceClassification.from_pretrained(m["base"], num_labels=m["num_labels"])

args = DPOConfig(
    output_dir=str(resolve(t["output_dir"])),
    beta=t["beta"],
    learning_rate=t["learning_rate"],
    per_device_train_batch_size=t["per_device_train_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    num_train_epochs=t["num_train_epochs"],
    warmup_ratio=t["warmup_ratio"],
    max_grad_norm=t["max_grad_norm"],
    seed=t["seed"],
    max_length=m["max_length"],
    logging_steps=10,
    report_to=[],
)
print(args.output_dir)

In [ ]:
import mlflow
mlflow.set_tracking_uri(rcfg["mlflow"]["tracking_uri"])
mlflow.set_experiment(rcfg["mlflow"]["experiment"])

with mlflow.start_run(run_name="dpo_v1") as run:
    mlflow.log_params({
        "base_model": m["base"], "beta": t["beta"], "lr": t["learning_rate"],
        "epochs": t["num_train_epochs"], "n_pairs": len(pairs),
        "min_score_gap": rcfg["pairs"]["min_score_gap"],
    })
    trainer = DPOTrainer(model=model, ref_model=ref_model, args=args,
                         train_dataset=ds["train"], eval_dataset=ds["test"],
                         processing_class=tok)
    trainer.train()
    trainer.save_model(args.output_dir)
    run_id = run.info.run_id
print("saved to", args.output_dir, "run", run_id)

### The only evaluation that counts

Re-run notebook 06 with the tuned model selected, and compare against the baseline on
the same held-out queries. Log the deltas to the same MLflow run so the model and its
score live together.

In [ ]:
# Point the retriever at the tuned model and re-measure.
# vcfg["rerank"]["active"] = "dpo"  -> resolved through reranker/registry_B.py
#
# from research_assistant.retrieval.service_B import search
# df_dpo, dpo = evaluate(search, label="hybrid + DPO reranker")
# delta = {k: round(dpo[k] - base[k], 4) for k in base}
# with mlflow.start_run(run_id=run_id):
#     mlflow.log_metrics({f"eval_{k}": float(v) for k, v in dpo.items()})
#     mlflow.log_metrics({f"delta_{k}": float(v) for k, v in delta.items()})
# delta

## Exit checks

- [ ] The tuned model scores on the same held-out set as the baseline, same code path.
- [ ] Both runs are in MLflow and comparable side by side.
- [ ] You checked latency. A cross-encoder over 20 candidates adds real milliseconds,
      and that cost belongs in the report next to the gain.
- [ ] If it lost to the baseline, the registry still points at the baseline, and the
      ledger records the negative result rather than hiding it.